# Wave 1 — 1D: Validasi Independen Wave 1
Menggabungkan output **Module A, B, C** (dari folder `wave 0.A`, `wave 0.B`, `wave 0.C`) lalu memvalidasinya terhadap `ground_truth.csv`.

Checklist:
1. **Tabel gabungan**: `employer_id` → skor & status A, B, C
2. **Bandingkan dengan ground truth**: berapa kasus yang di-inject berhasil di-flag oleh **modul yang seharusnya**
3. **Cek employer bersih**: tidak boleh ada yang di-flag di ketiga modul sekaligus tanpa alasan (indikasi bug)
4. **Definition of done Wave 1**: tiga modul masing-masing mengeluarkan skor + status per employer dan sudah tervalidasi kasar

**Jalankan notebook A, B, C dulu** (supaya file `module_*_scores.csv` ada di Drive), lalu Runtime → Run all di sini. Output ke folder `wave 0.D`.

In [ ]:
import os, glob
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 140); pd.set_option("display.width", 220)
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

## 1. Cari output Module A, B, C + ground truth di Drive

In [ ]:
BASE_DIR  = "/content/drive/MyDrive/Healthkathon Engine"
DRIVE_DIR = f"{BASE_DIR}/Dummy Healthkathon"
OUT_DRIVE = f"{BASE_DIR}/wave 0.D"

if os.environ.get("HK_WAVE1_DIR"):                      # testing lokal
    SEARCH_ROOT, DATA_DIR = Path(os.environ["HK_WAVE1_DIR"]), Path(os.environ["HK_DATA_DIR"])
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SEARCH_ROOT, DATA_DIR = Path("/content/drive/MyDrive"), Path(DRIVE_DIR)
else:
    SEARCH_ROOT, DATA_DIR = Path("."), Path("data")
OUT_DIR = Path(os.environ.get("HK_OUT_DIR", OUT_DRIVE if IN_COLAB else "output_wave1_validation"))

def find_latest(filename):
    # cari di seluruh Drive (nama persis, jadi file 'Salinan ...' tidak ikut), ambil yang paling baru
    hits = [Path(p) for p in glob.glob(str(SEARCH_ROOT / "**" / filename), recursive=True)]
    if not hits:
        return None
    return max(hits, key=lambda p: p.stat().st_mtime)

FILES = {m: find_latest(f"module_{m.lower()}_scores.csv") for m in ["A", "B", "C"]}
GT_PATH = DATA_DIR / "ground_truth.csv"
if not GT_PATH.exists():
    GT_PATH = find_latest("ground_truth.csv")

for m, p in FILES.items():
    if p is None:
        print(f"❌ Module {m}: module_{m.lower()}_scores.csv TIDAK ditemukan — jalankan notebook Module {m} dulu")
    else:
        ts = datetime.fromtimestamp(p.stat().st_mtime).strftime("%Y-%m-%d %H:%M")
        print(f"✓ Module {m}: {p}  (diubah {ts})")
print(f"✓ ground_truth: {GT_PATH}")

## 2. Tabel gabungan: employer_id → skor & status A, B, C
- `score_X` = skor mentah modul (satuan asli modul), `score_X_norm` = skor 0–1 dari notebook modulnya
- `NO_DATA` = employer tidak ada di output modul itu (misal tidak punya data setoran)

In [ ]:
VALID_STATUS = {
    "A": {"FLAGGED", "NORMAL", "EXPLAINED_BY_RESIGN", "INSUFFICIENT_DATA"},
    "B": {"FLAGGED", "NORMAL", "NOT_CANDIDATE", "INSUFFICIENT_DATA"},
    "C": {"FLAGGED", "NORMAL", "ONE_OFF_GAP", "INSUFFICIENT_DATA"},
}
MODULE_LABEL = {"A": "Sembunyiin karyawan", "B": "Lapor gaji lebih rendah", "C": "Setoran tidak sesuai"}

RAW = {}
for m, p in FILES.items():
    if p is None:
        continue
    df = pd.read_csv(p)
    df["employer_id"] = df["employer_id"].astype(str)
    RAW[m] = df

frames = []
for m, df in RAW.items():
    k = m.lower()
    keep = {"employer_id": "employer_id", "status": f"status_{m}", f"score_{k}": f"score_{m}",
            f"score_{k}_norm": f"score_{m}_norm", "reason": f"reason_{m}"}
    frames.append(df[[c for c in keep if c in df.columns]].rename(columns=keep).set_index("employer_id"))
COMBINED = pd.concat(frames, axis=1).reset_index()
for m in RAW:
    COMBINED[f"status_{m}"] = COMBINED[f"status_{m}"].fillna("NO_DATA")
COMBINED["n_flags"] = sum(COMBINED[f"status_{m}"].eq("FLAGGED").astype(int) for m in RAW)
COMBINED["flagged_by"] = COMBINED.apply(
    lambda r: "+".join(m for m in RAW if r[f"status_{m}"] == "FLAGGED") or "-", axis=1)

print(f"Employer di tabel gabungan: {len(COMBINED):,}")
for m in RAW:
    print(f"  Module {m}: {len(RAW[m]):,} employer | " +
          ", ".join(f"{k}={v}" for k, v in COMBINED[f'status_{m}'].value_counts().items()))
COMBINED.sort_values("n_flags", ascending=False).head(10)[
    ["employer_id", "flagged_by"] + [c for c in COMBINED.columns if c.startswith(("status_", "score_")) and not c.endswith("_norm")]]

## 3. Cek konsistensi output (indikasi bug)
Hal-hal yang **tidak boleh terjadi** kalau modul benar:

In [ ]:
BUGS = []
def check(name, ok, detail=""):
    print(("✅" if ok else "❌"), name, ("" if ok else f"-> {detail}"))
    if not ok:
        BUGS.append(name)

for m, df in RAW.items():
    k = m.lower()
    check(f"[{m}] employer_id unik", df["employer_id"].is_unique,
          f"{df['employer_id'].duplicated().sum()} duplikat")
    bad = set(df["status"]) - VALID_STATUS[m]
    check(f"[{m}] semua status valid", not bad, f"status tak dikenal: {bad}")
    fl = df[df["status"] == "FLAGGED"]
    check(f"[{m}] semua FLAGGED punya skor > 0", (fl[f"score_{k}"] > 0).all(),
          f"{(fl[f'score_{k}'] <= 0).sum()} baris")
    check(f"[{m}] semua FLAGGED punya alasan", fl["reason"].fillna("").str.len().gt(0).all())
    ins = df[df["status"] == "INSUFFICIENT_DATA"]
    check(f"[{m}] INSUFFICIENT_DATA tidak diberi skor (netral)", ins[f"score_{k}"].isna().all(),
          f"{ins[f'score_{k}'].notna().sum()} baris punya skor")
    if f"score_{k}_norm" in df:
        check(f"[{m}] skor ternormalisasi dalam 0–1", df[f"score_{k}_norm"].dropna().between(0, 1).all())
    check(f"[{m}] tidak ada kolom data pribadi",
          not ({"nik", "nama", "nama_karyawan", "no_ktp"} & {c.lower() for c in df.columns}))
print(f"\nJumlah masalah konsistensi: {len(BUGS)}")

## 4. Bandingkan dengan ground truth
Setiap label di ground truth dipetakan ke **modul yang seharusnya menangkapnya**. Cek `LABEL_TO_MODULE`: kalau ada label yang tercetak `(tidak dikenal)`, tambahkan kata kuncinya.

In [ ]:
LABEL_TO_MODULE = {
    "A": ["PDUK", "HIDDEN", "HEADCOUNT", "EMPLOYEE", "KARYAWAN", "SEMBUNYI"],
    "B": ["WAGE", "UPAH", "GAJI", "SALARY", "DPI", "UNDERPAY"],
    "C": ["REMITTANCE", "SETOR", "IURAN", "CONTRIBUTION", "PENGGELAPAN"],
}
CLEAN_LABELS = {"CLEAN", "NONE", "NORMAL", "-", "NAN"}

GT = pd.read_csv(GT_PATH)
gt_emp = next(c for c in GT.columns if c.lower() in ("employer_id", "id_employer", "company_id", "npp"))
gt_lab = next(c for c in GT.columns if c.lower() in ("anomaly_type", "fraud_type", "label", "scenario", "jenis_fraud"))
GT = GT.rename(columns={gt_emp: "employer_id", gt_lab: "label"})[["employer_id", "label"]]
GT["employer_id"] = GT["employer_id"].astype(str)
GT["label"] = GT["label"].astype(str).str.upper().str.strip()

def expected_module(lab):
    if lab in CLEAN_LABELS:
        return "CLEAN"
    hits = [m for m, pats in LABEL_TO_MODULE.items() if any(p in lab for p in pats)]
    return "+".join(hits) if hits else "(tidak dikenal)"

GT["expected_module"] = GT["label"].map(expected_module)
print("Label → modul yang seharusnya:")
print(GT.groupby(["label", "expected_module"]).size().rename("n").reset_index().to_string(index=False))

V = GT.merge(COMBINED, on="employer_id", how="left")
for m in RAW:
    V[f"status_{m}"] = V[f"status_{m}"].fillna("NO_DATA")
V["n_flags"] = V["n_flags"].fillna(0).astype(int); V["flagged_by"] = V["flagged_by"].fillna("-")

In [ ]:
def per_label_summary(v):
    rows = []
    for lab, g in v.groupby("label"):
        exp = g["expected_module"].iloc[0]
        if exp in ("CLEAN", "(tidak dikenal)"):
            continue
        mods = [m for m in exp.split("+") if m in RAW]
        right = g[[f"status_{m}" for m in mods]].eq("FLAGGED").any(axis=1)
        any_flag = g["n_flags"] > 0
        not_judged = g[[f"status_{m}" for m in mods]].isin(["INSUFFICIENT_DATA", "NO_DATA"]).all(axis=1)
        rows.append({
            "label": lab, "modul_seharusnya": exp, "n_kasus": len(g),
            "ke-flag_modul_benar": int(right.sum()),
            "recall_modul_benar": round(float(right.mean()), 3),
            "hanya_ke-flag_modul_lain": int((any_flag & ~right).sum()),
            "tidak_ke-flag_sama_sekali": int((~any_flag & ~not_judged).sum()),
            "belum_bisa_dinilai": int((~right & not_judged).sum()),
        })
    return pd.DataFrame(rows)

LABEL_SUMMARY = per_label_summary(V)
print("=== Apakah kasus yang di-inject ke-flag oleh modul yang seharusnya? ===")
display(LABEL_SUMMARY) if "display" in globals() else print(LABEL_SUMMARY.to_string(index=False))

print("\n=== Label ground truth × kombinasi modul yang mem-flag ===")
CROSSTAB = pd.crosstab(V["label"], V["flagged_by"], margins=True, margins_name="total")
display(CROSSTAB) if "display" in globals() else print(CROSSTAB.to_string())

In [ ]:
# Detail kasus yang terlewat oleh modul yang seharusnya
is_expected = lambda s, m: s.str.split("+").apply(lambda mods: m in mods)
for m in RAW:
    miss = V[is_expected(V["expected_module"], m) & V[f"status_{m}"].ne("FLAGGED")]
    if len(miss):
        print(f"\n--- Kasus Module {m} ({MODULE_LABEL[m]}) yang TIDAK ke-flag: {len(miss)} ---")
        cols = ["employer_id", "label", f"status_{m}", f"score_{m}", f"reason_{m}"]
        print(miss[[c for c in cols if c in miss]].head(10).to_string(index=False))

## 5. Cek employer bersih
- Di-flag **3 modul sekaligus** padahal bersih → hampir pasti **bug** (misal join salah, satuan tertukar).
- Di-flag **2 modul** → perlu dilihat alasannya satu per satu.
- Di-flag 1 modul → false positive biasa, sudah terukur di FPR tiap modul.

In [ ]:
CLEAN = V[V["expected_module"] == "CLEAN"]
print(f"Employer bersih: {len(CLEAN):,}")
print(CLEAN["n_flags"].value_counts().sort_index().rename_axis("jumlah modul yang mem-flag").rename("n_employer").to_string())

fpr = {}
for m in RAW:
    judged = CLEAN[~CLEAN[f"status_{m}"].isin(["INSUFFICIENT_DATA", "NO_DATA"])]
    fpr[m] = round(float((judged[f"status_{m}"] == "FLAGGED").mean()), 3) if len(judged) else None
print("\nFalse positive rate per modul (di employer bersih):", fpr)

ALL3 = CLEAN[CLEAN["n_flags"] >= len(RAW)] if len(RAW) == 3 else CLEAN.iloc[0:0]
TWO = CLEAN[CLEAN["n_flags"] == 2]
check("Tidak ada employer bersih yang di-flag di ketiga modul sekaligus", len(ALL3) == 0,
      f"{len(ALL3)} employer: {ALL3['employer_id'].tolist()[:10]}")
if len(ALL3):
    print(ALL3[["employer_id"] + [f"reason_{m}" for m in RAW]].to_string(index=False))
if len(TWO):
    print(f"\n⚠ {len(TWO)} employer bersih di-flag 2 modul — cek alasannya:")
    print(TWO[["employer_id", "flagged_by"] + [f"reason_{m}" for m in RAW]].head(10).to_string(index=False))

## 6. Definition of Done Wave 1

In [ ]:
DOD = []
for m in ["A", "B", "C"]:
    has = m in RAW
    ok_cols = has and {"employer_id", "status", f"score_{m.lower()}"} <= set(RAW[m].columns)
    row = LABEL_SUMMARY[is_expected(LABEL_SUMMARY["modul_seharusnya"], m)] if has else pd.DataFrame()
    rec = round(row["ke-flag_modul_benar"].sum() / max(row["n_kasus"].sum() - row["belum_bisa_dinilai"].sum(), 1), 3) \
        if len(row) else np.nan
    DOD.append(dict(modul=m, nama=MODULE_LABEL[m], output_ada=has, skor_dan_status=bool(ok_cols),
                    recall_vs_ground_truth=rec, false_positive_rate=fpr.get(m, np.nan)))
DOD = pd.DataFrame(DOD)
display(DOD) if "display" in globals() else print(DOD.to_string(index=False))

done = DOD["output_ada"].all() and DOD["skor_dan_status"].all() and not BUGS
print("\n" + ("✅ Definition of done Wave 1 TERPENUHI" if done else "❌ Belum terpenuhi") +
      f" — {len(BUGS)} masalah konsistensi" + (f": {BUGS}" if BUGS else ""))
print("Catatan: recall yang rendah bukan bug, tapi bahan tuning threshold di notebook modulnya.")

## 7. Simpan hasil validasi ke Drive (`wave 0.D`)

In [ ]:
if IN_COLAB and str(OUT_DIR).startswith("/content/drive") and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
OUT_DIR.mkdir(parents=True, exist_ok=True)
V.to_csv(OUT_DIR / "wave1_combined_scores.csv", index=False)
LABEL_SUMMARY.to_csv(OUT_DIR / "wave1_recall_per_label.csv", index=False)
CROSSTAB.to_csv(OUT_DIR / "wave1_label_vs_flags.csv")
DOD.to_csv(OUT_DIR / "wave1_definition_of_done.csv", index=False)
with open(OUT_DIR / "wave1_validation_report.txt", "w") as f:
    f.write(f"Validasi Wave 1 — {datetime.now():%Y-%m-%d %H:%M}\n\n")
    f.write("Sumber:\n" + "".join(f"  Module {m}: {p}\n" for m, p in FILES.items()) + "\n")
    f.write(LABEL_SUMMARY.to_string(index=False) + "\n\n")
    f.write(f"FPR per modul: {fpr}\n")
    f.write(f"Employer bersih di-flag 3 modul: {len(ALL3)} | 2 modul: {len(TWO)}\n")
    f.write(f"Masalah konsistensi: {BUGS or 'tidak ada'}\n\n")
    f.write(DOD.to_string(index=False) + "\n")
print("Tersimpan di:", OUT_DIR)
for f in sorted(OUT_DIR.glob("wave1_*")):
    print("  ✓", f.name)